# Synthetic Datasets: Dataset Utility Metrics

Author: Ilse Harmers \
Last modified: February 26, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sdmetrics.single_column import TVComplement
from scipy.stats import wasserstein_distance
from scipy.stats import kstest

## Initialization

In [ ]:
# Initializing the real datasets (Adult, Bank Marketing and Credit Card Default) and our functions. 
adult_train = pd.read_csv("./train-test-datasets/adult/adult_train.csv")
credit_train = pd.read_csv("./train-test-datasets/credit-card-default/credit_train.csv")
bank_train = pd.read_csv("./train-test-datasets/bank-marketing/bank_train.csv")

# Epsilon values.
epsi = [1, 2, 5, 8]

def encode_adult(sample):
    """This function encodes an Adult-based synthetic dataset such that all numerical columns are normalized to a range of (0, 1)."""

    sample["age"] = ((sample["age"] - sample["age"].min()) / (sample["age"].max() - sample["age"].min()))
    sample["education-num"] = ((sample["education-num"] - sample["education-num"].min()) / 
                               (sample["education-num"].max() - sample["education-num"].min()))
    sample["capital-gain"] = ((sample["capital-gain"] - sample["capital-gain"].min()) / 
                              (sample["capital-gain"].max() - sample["capital-gain"].min()))
    sample["capital-loss"] = ((sample["capital-loss"] - sample["capital-loss"].min()) / 
                              (sample["capital-loss"].max() - sample["capital-loss"].min()))
    sample["hours-per-week"] = ((sample["hours-per-week"] - sample["hours-per-week"].min()) / 
                                (sample["hours-per-week"].max() - sample["hours-per-week"].min()))

    return sample

def encode_bank(sample):
    """This function encodes a Bank-based synthetic dataset such that all numerical columns are normalized to a range of (0, 1)."""

    sample["age"] = ((sample["age"] - sample["age"].min()) / (sample["age"].max() - sample["age"].min()))
    sample["balance"] = ((sample["balance"] - sample["balance"].min()) / (sample["balance"].max() - sample["balance"].min()))
    sample["day"] = ((sample["day"] - sample["day"].min()) / (sample["day"].max() - sample["day"].min()))
    sample["duration"] = ((sample["duration"] - sample["day"].min()) / 
                          (sample["duration"].max() - sample["duration"].min()))
    sample["campaign"] = ((sample["campaign"] - sample["campaign"].min()) / 
                          (sample["campaign"].max() - sample["campaign"].min()))
    sample["pdays"] = ((sample["pdays"] - sample["pdays"].min()) / (sample["pdays"].max() - sample["pdays"].min()))
    sample["previous"] = ((sample["previous"] - sample["previous"].min()) / 
                          (sample["previous"].max() - sample["previous"].min()))

    return sample
    
def encode_credit(sample):
    """This function encodes a Credit-based synthetic dataset such that all numerical columns are normalized to a range of (0, 1)."""

    sample["SEX"] = ((sample["SEX"] - sample["SEX"].min()) / (sample["SEX"].max() - sample["SEX"].min()))
    sample["AGE"] = ((sample["AGE"] - sample["AGE"].min()) / (sample["AGE"].max() - sample["AGE"].min()))
    sample["LIMIT_BAL"] = ((sample["LIMIT_BAL"] - sample["LIMIT_BAL"].min()) / 
                           (sample["LIMIT_BAL"].max() - sample["LIMIT_BAL"].min()))
    sample["EDUCATION"] = ((sample["EDUCATION"] - sample["EDUCATION"].min()) / 
                           (sample["EDUCATION"].max() - sample["EDUCATION"].min()))
    sample["MARRIAGE"] = ((sample["MARRIAGE"] - sample["MARRIAGE"].min()) / 
                          (sample["MARRIAGE"].max() - sample["MARRIAGE"].min()))
    
    for p in [0, 2, 3, 4, 5, 6]:
        sample[f"PAY_{p}"] = ((sample[f"PAY_{p}"] - sample[f"PAY_{p}"].min()) / 
                              (sample[f"PAY_{p}"].max() - sample[f"PAY_{p}"].min()))

    for a in range(1, 7):
        sample[f"BILL_AMT{a}"] = ((sample[f"BILL_AMT{a}"] - sample[f"BILL_AMT{a}"].min()) / 
                                  (sample[f"BILL_AMT{a}"].max() - sample[f"BILL_AMT{a}"].min()))
        sample[f"PAY_AMT{a}"] = ((sample[f"PAY_AMT{a}"] - sample[f"PAY_AMT{a}"].min()) / 
                                 (sample[f"PAY_AMT{a}"].max() - sample[f"PAY_AMT{a}"].min()))

    return sample
    
def KS_metric(num_cols, real_dataset, synthetic_dataset):
    """This function computes the Kolmogorov-Smirnov statistic for each numerical column in the real and synthetic dataset."""
    
    KS_vals = {}
    p_vals = {}
    for n in num_cols:
        KS_test = kstest(real_dataset[n], synthetic_dataset[n])
        KS_vals[n] = KS_test[0]
        p_vals[n] = KS_test[1]
    if not (np.array(list(p_vals.values())) <= 0.05).all():
        print("There is a p-value larger than 0.05: ", p_vals)
    return (KS_vals, np.mean(list(KS_vals.values())), np.std(list(KS_vals.values())))

def TV_metric(cat_cols, real_dataset, synthetic_dataset):
    """This function computes the total variation distance (TVD) for each categorical column in the real and synthetic dataset."""
    
    TV_vals = {}
    for c in cat_cols:
        TV_val = TVComplement.compute(real_data = real_dataset[c], synthetic_data = synthetic_dataset[c])
        TV_vals[c] = 1 - TV_val   # reversing the order provided by sdmetrics package; now '0' = 'high similarity' and '>> 0' = 'low similarity' 
    return (TV_vals, np.mean(list(TV_vals.values())), np.std(list(TV_vals.values())))

def wasserstein_metric(num_cols, real_dataset, synthetic_dataset):
    """This function computes the Wasserstein distance for each numerical column in the real and synthetic dataset."""
    
    ws_vals = {}
    for n in num_cols:
        ws_val = wasserstein_distance(real_dataset[n].values, synthetic_dataset[n].values)
        ws_vals[n] = ws_val
    return (ws_vals, np.mean(list(ws_vals.values())), np.std(list(ws_vals.values())))

def results_analysis(file_path, epsi, real_dataset, num_cols, cat_cols, dataset):
    """This function computes the dataset utility metrics computed for the real and synthetic datasets. The filepath that is provided
    serves as the place where the synthetic datasets are imported from. 
    
    If there are no categorical columns, set 'cat_cols' = []. 
    If the GAN model does not have privacy constraints, set 'epsi' = [].
    The 'dataset' parameter can be set to: "Adult", "Bank" or "Credit".
    """
    
    KS_results = {}
    if cat_cols:
        TV_results = {}
    WS_results = {}
    runs = range(1, 6)  
    samples = range(1, 4)
    c = 1
    for r in runs:
        for s in samples:
            if epsi:
                synth_data = pd.read_csv(file_path + f"epsi_{epsi}/run{r}/sample{s}.csv")
            else:
                synth_data = pd.read_csv(file_path + f"run{r}/sample{s}.csv")

            KS_syn = KS_metric(num_cols = num_cols, real_dataset = real_dataset, synthetic_dataset = synth_data)[0]
            KS_results[f"sample{c}"] = KS_syn

            if cat_cols:
                TV_syn = TV_metric(cat_cols = cat_cols, real_dataset = real_dataset, synthetic_dataset = synth_data)[0]
                TV_results[f"sample{c}"] = TV_syn
            
            # Min-max scaling the continuous columns.
            if dataset == "Adult":
                real = real_dataset.copy(deep = True)
                real_enc = encode_adult(real)
                synth_data = encode_adult(synth_data)
            elif dataset == "Bank":
                real = real_dataset.copy(deep = True)
                real_enc = encode_bank(real)
                synth_data = encode_bank(synth_data)
            elif dataset == "Credit":
                real = real_dataset.copy(deep = True)
                real_enc = encode_credit(real)
                synth_data = encode_credit(synth_data)
            else:
                print("Dataset not supported.")
                break
            
            WS_syn = wasserstein_metric(num_cols = num_cols, real_dataset = real_enc, synthetic_dataset = synth_data)[0]
            WS_results[f"sample{c}"] = WS_syn
            
            c += 1
    #print(WS_results)
    if cat_cols:
        return pd.DataFrame(KS_results), pd.DataFrame(TV_results), pd.DataFrame(WS_results)
    else:
        return pd.DataFrame(KS_results), pd.DataFrame(WS_results)

def print_results(file_path, epsilons, real_dataset, num_cols, cat_cols, dataset):
    """This function computes and stores the dataset utility metrics computed for the real and synthetic datasets. The filepath that is provided
    will simultaneously serve as the place where the synthetic datasets are imported from and where the dataset utility results are stored. 
    
    If there are no categorical columns, set 'cat_cols' = []. 
    The 'dataset' parameter can be set to: "Adult", "Bank" or "Credit".
    """

    for e in epsilons:
        if cat_cols:
            KS_results, TV_results, WS_results = results_analysis(file_path = file_path, epsi = e, real_dataset = real_dataset, num_cols = num_cols,
                                                                  cat_cols = cat_cols, dataset = dataset)

            sample_results_mean = [np.mean(KS_results, axis = 0), np.mean(TV_results, axis = 0), np.mean(WS_results, axis = 0)]
            results_mean = pd.concat(sample_results_mean, axis = 1).rename(columns = {0: "KS", 1: "TVD", 2: "WSS"})
            results_mean.to_csv(file_path + f"dataset-utility-results_mean_epsi-{e}.csv")

            sample_results_std = [np.std(KS_results, axis = 0), np.std(TV_results, axis = 0), np.std(WS_results, axis = 0)]
            results_std = pd.concat(sample_results_std, axis = 1).rename(columns = {0: "KS", 1: "TVD", 2: "WSS"})
            results_std.to_csv(file_path + f"dataset-utility-results_std_epsi-{e}.csv")
            
            print(f"Epsilon = {e}")
            print("K-S results")
            print(np.mean(KS_results), np.std(KS_results.values))
            print("WSS results")
            print(np.mean(WS_results), np.std(WS_results.values))
            print("TVD results")
            print(np.mean(TV_results), np.std(TV_results.values))
            print()
        else:
            KS_results, WS_results = results_analysis(file_path = file_path, epsi = e, real_dataset = real_dataset, num_cols = num_cols,
                                                      cat_cols = cat_cols, dataset = dataset)

            sample_results_mean = [np.mean(KS_results, axis = 0), np.mean(WS_results, axis = 0)]
            results_mean = pd.concat(sample_results_mean, axis = 1).rename(columns = {0: "KS", 1: "WSS"})
            results_mean.to_csv(file_path + f"dataset-utility-results_mean_epsi-{e}.csv")

            sample_results_std = [np.std(KS_results, axis = 0), np.std(WS_results, axis = 0)]
            results_std = pd.concat(sample_results_std, axis = 1).rename(columns = {0: "KS", 1: "WSS"})
            results_std.to_csv(file_path + f"dataset-utility-results_std_epsi-{e}.csv")
            
            print(f"Epsilon = {e}")
            print("K-S results")
            print(np.mean(KS_results), np.std(KS_results.values))
            print("WSS results")
            print(np.mean(WS_results), np.std(WS_results.values))
            print()
            

## Adult Dataset (Real)

In [ ]:
# Initializing the categorical and numerical columns of the Adult dataset.
cat_cols = adult_train.select_dtypes(include=['object']).columns.to_list()
num_cols = adult_train.select_dtypes(exclude=['object']).columns.to_list()

In [ ]:
# Sanity check of Wasserstein metric.
print("adult_train (WSS mean, WSS std): ", wasserstein_metric(num_cols = num_cols, real_dataset = adult_train, synthetic_dataset = adult_train)[1:])
print("adult_train (WSS mean, WSS std): ", wasserstein_metric(num_cols = num_cols, real_dataset = adult_train, 
                                                              synthetic_dataset = adult_train.iloc[::-1])[1:])
print(wasserstein_metric(num_cols = num_cols, real_dataset = adult_train, synthetic_dataset = adult_train)[0])

## Synthetic Adult Dataset (TabFairGAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_TabFair/adult/"
KS1_results, TV1_results, WS1_results = results_analysis(file_path = file_path, epsi = [], real_dataset = adult_train, num_cols = num_cols,
                                                         cat_cols = cat_cols, dataset = "Adult")

print("K-S results")
print(np.mean(KS1_results), np.std(KS1_results.values))
print("WSS results")
print(np.mean(WS1_results), np.std(WS1_results.values))
print("TVD results")
print(np.mean(TV1_results), np.std(TV1_results.values))

sample_results_mean = [np.mean(KS1_results, axis = 0), np.mean(TV1_results, axis = 0), np.mean(WS1_results, axis = 0)]
results_mean = pd.concat(sample_results_mean, axis = 1).rename(columns = {0: "KS", 1: "TVD", 2: "WSS"})
results_mean.to_csv(file_path + "dataset-utility-results_mean.csv")

sample_results_std = [np.std(KS1_results, axis = 0), np.std(TV1_results, axis = 0), np.std(WS1_results, axis = 0)]
results_std = pd.concat(sample_results_std, axis = 1).rename(columns = {0: "KS", 1: "TVD", 2: "WSS"})
results_std.to_csv(file_path + "dataset-utility-results_std.csv")

## Synthetic Adult Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_DP-GAN/adult/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = adult_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Adult")

## Synthetic Adult Dataset (DP-CTGAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "synthetic-datasets_DPCTGAN/adult/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = adult_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Adult")

## Synthetic Adult Dataset (Fair DP-GAN with Demographic Parity Loss)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "synthetic-datasets_FairDP-GAN(dem)/adult/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = adult_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Adult")

## Synthetic Adult Dataset (Fair DP-GAN with Disparate Impact Loss)

In [ ]:
# Assigning file path for importing the dataset. 
file_path = "synthetic-datasets_FairDP-GAN(dis)/adult/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = adult_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Adult")

## Synthetic Adult Dataset (Fair DP-GAN with All Fair Losses)

In [ ]:
# Assigning file path for importing the dataset. 
file_path = "synthetic-datasets_FairDP-GAN(dem-dis)/adult/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = adult_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Adult")

## Bank Marketing Dataset (Real)

In [ ]:
# Initializing the categorical and numerical columns of the Bank dataset.
cat_cols = bank_train.select_dtypes(include=['object']).columns.to_list()
num_cols = bank_train.select_dtypes(exclude=['object']).columns.to_list()

In [ ]:
# Sanity check of Wasserstein metric.
print("bank_train (WSS mean, WSS std): ", wasserstein_metric(num_cols = num_cols, real_dataset = bank_train, synthetic_dataset = bank_train)[1:])
print("bank_train (WSS mean, WSS std): ", wasserstein_metric(num_cols = num_cols, real_dataset = bank_train, 
                                                             synthetic_dataset = bank_train.iloc[::-1])[1:])
print(wasserstein_metric(num_cols = num_cols, real_dataset = bank_train, synthetic_dataset = bank_train)[0])

## Synthetic Bank Marketing Dataset (TabFairGAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "synthetic-datasets_TabFair/bank-marketing/"
KS1_results, TV1_results, WS1_results = results_analysis(file_path = file_path, epsi = [], real_dataset = bank_train, num_cols = num_cols,
                                                         cat_cols = cat_cols, dataset = "Bank")

print("K-S results")
print(np.mean(KS1_results), np.std(KS1_results.values))
print("WSS results")
print(np.mean(WS1_results), np.std(WS1_results.values))
print("TVD results")
print(np.mean(TV1_results), np.std(TV1_results.values))

sample_results_mean = [np.mean(KS1_results, axis = 0), np.mean(TV1_results, axis = 0), np.mean(WS1_results, axis = 0)]
results_mean = pd.concat(sample_results_mean, axis = 1).rename(columns = {0: "KS", 1: "TVD", 2: "WSS"})
results_mean.to_csv(file_path + "dataset-utility-results_mean.csv")

sample_results_std = [np.std(KS1_results, axis = 0), np.std(TV1_results, axis = 0), np.std(WS1_results, axis = 0)]
results_std = pd.concat(sample_results_std, axis = 1).rename(columns = {0: "KS", 1: "TVD", 2: "WSS"})
results_std.to_csv(file_path + "dataset-utility-results_std.csv")

## Synthetic Bank Marketing Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_DP-GAN/bank-marketing/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = bank_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Bank")

## Synthetic Bank Marketing Dataset (DP-CTGAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "synthetic-datasets_DPCTGAN/bank-marketing/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = bank_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Bank")

## Synthetic Bank Marketing Dataset (Fair DP-GAN with Demographic Parity Loss)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_FairDP-GAN(dem)/bank-marketing/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = bank_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Bank")

## Synthetic Bank Marketing Dataset (Fair DP-GAN with Disparate Impact Loss)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_FairDP-GAN(dis)/bank-marketing/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = bank_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Bank")

## Synthetic Bank Marketing Dataset (Fair DP-GAN with All Fair Losses)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_FairDP-GAN(dem-dis)/bank-marketing/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = bank_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Bank")

## Credit Card Default Dataset (Real)

In [ ]:
# Initializing the categorical and numerical columns of the Credit dataset.
cat_cols = credit_train.select_dtypes(include=['object']).columns.to_list()
num_cols = credit_train.select_dtypes(exclude=['object']).columns.to_list()

In [ ]:
# Sanity check of Wasserstein metric.
print("credit_train (WSS mean, WSS std): ", wasserstein_metric(num_cols = num_cols, real_dataset = credit_train, synthetic_dataset = credit_train)[1:])
print("credit_train (WSS mean, WSS std): ", wasserstein_metric(num_cols = num_cols, real_dataset = credit_train, 
                                                             synthetic_dataset = credit_train.iloc[::-1])[1:])
print(wasserstein_metric(num_cols = num_cols, real_dataset = credit_train, synthetic_dataset = credit_train)[0])

## Synthetic Credit Card Default Dataset (TabFairGAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_TabFair/credit-card-default/"
KS1_results, WS1_results = results_analysis(file_path = file_path, epsi = [], real_dataset = credit_train, num_cols = num_cols,
                                                         cat_cols = cat_cols, dataset = "Credit")

print("K-S results")
print(np.mean(KS1_results), np.std(KS1_results.values))
print("WSS results")
print(np.mean(WS1_results), np.std(WS1_results.values))

sample_results_mean = [np.mean(KS1_results, axis = 0), np.mean(WS1_results, axis = 0)]
results_mean = pd.concat(sample_results_mean, axis = 1).rename(columns = {0: "KS", 1: "WSS"})
results_mean.to_csv(file_path + "dataset-utility-results_mean.csv")

sample_results_std = [np.std(KS1_results, axis = 0), np.std(WS1_results, axis = 0)]
results_std = pd.concat(sample_results_std, axis = 1).rename(columns = {0: "KS", 1: "WSS"})
results_std.to_csv(file_path + "dataset-utility-results_std.csv")

## Synthetic Credit Card Default Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_DP-GAN/credit-card-default/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = credit_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Credit")

## Synthetic Credit Card Default Dataset (DP-CTGAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "synthetic-datasets_DPCTGAN/credit-card-default/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = credit_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Credit")

## Synthetic Credit Card Default Dataset (Fair DP-GAN with Demographic Parity Loss)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_FairDP-GAN(dem)/credit-card-default/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = credit_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Credit")

## Synthetic Credit Card Default Dataset (Fair DP-GAN with Disparate Impact Loss)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_FairDP-GAN(dis)/credit-card-default/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = credit_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Credit")

## Synthetic Credit Card Default Dataset (Fair DP-GAN with All Fair Losses)

In [ ]:
# Assigning file path for importing the datasets. 
file_path = "synthetic-datasets_FairDP-GAN(dem-dis)/credit-card-default/"
print_results(file_path = file_path, epsilons = [1, 2, 5, 8], real_dataset = credit_train, num_cols = num_cols, cat_cols = cat_cols, dataset = "Credit")